In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("C:/Users/ok/Documents/FlyRank Internship/Week 1/flyrank-ml-internship-starter-main/data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## Choose two signals

In [5]:
signal1 = "avg_position"
signal2 = "content_age_days"

## Analyse Signal 1

In [6]:
pd.cut(df["avg_position"], bins=[0,10,20,50,100]).value_counts().sort_index()

avg_position
(0, 10]      12983
(10, 20]      7273
(20, 50]      7225
(50, 100]     1299
Name: count, dtype: int64


Pages with poorer average positions may have greater opportunity for optimisation, making this a useful prioritisation signal.

## Analyse Signal 2

In [7]:
pd.cut(df["content_age_days"], bins=5).value_counts().sort_index()

content_age_days
(89.526, 184.8]    12558
(184.8, 279.6]      4361
(279.6, 374.4]      6757
(374.4, 469.2]      3713
(469.2, 564.0]      2611
Name: count, dtype: int64

Older content is more likely to benefit from review or refresh.

## Baseline Rule

In [8]:
df["baseline_score"] = (
    df["avg_position"]*2
    + df["content_age_days"]*0.05
    - df["engagement_rate"]*20
)

In [9]:
def action(score):
    if score >= 80:
        return "Refresh"
    elif score >= 50:
        return "Review"
    else:
        return "Leave"

df["action"] = df["baseline_score"].apply(action)

In [10]:
def reason(row):
    if row["content_age_days"] > 300:
        return "Old content"
    elif row["avg_position"] > 20:
        return "Low ranking"
    else:
        return "Healthy page"

df["reason"] = df.apply(reason, axis=1)

In [11]:
ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

In [13]:
ranked.to_csv(
    "C:/Users/ok/Documents/FlyRank Internship/Week 1/flyrank-ml-internship-starter-main/work/outputs/baseline_action_score.csv",
    index=False
)

## Review top 10

In [14]:
top10 = ranked.head(10)

top10[
    [
        "baseline_score",
        "action",
        "reason"
    ]
]

,baseline_score,action,reason
24445,505.55,Refresh,Old content
19920,385.35,Refresh,Old content
26873,342.50,Refresh,Low ranking
16044,337.55,Refresh,Old content
18532,307.05,Refresh,Old content
15639,306.35,Refresh,Old content
27923,295.50,Refresh,Low ranking
2970,293.25,Refresh,Old content
28214,253.35,Refresh,Old content
1534,244.40,Refresh,Old content


| Page | Review                                                                                                                |
| ---- | --------------------------------------------------------------------------------------------------------------------- |
| 1    | This page has an old publication date and poor ranking, making it a strong candidate for a content refresh.           |
| 2    | The page shows a high baseline score due to its age and search position; updating the content may improve visibility. |
| 3    | This page should be reviewed because its ranking indicates potential for optimisation.                                |
| 4    | The content is relatively old, suggesting it may benefit from refreshed information.                                  |
| 5    | Low search performance and content age make this page a suitable refresh candidate.                                   |
| 6    | This page should be monitored for improvements after content updates.                                                 |
| 7    | The baseline score indicates moderate priority for review.                                                            |
| 8    | Updating this page could improve search performance over time.                                                        |
| 9    | The combination of age and ranking suggests an opportunity for optimisation.                                          |
| 10   | This page has been prioritised for review based on the baseline scoring rule.                                         |
